In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()

device = "cuda"  # the device to load the model onto

model_name = "speakleash/Bielik-11B-v2.2-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [2]:
model.to(device)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32128, 4096)
    (layers): ModuleList(
      (0-49): 50 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )
    (norm): MistralRMSNorm(

In [3]:
model.get_memory_footprint()

22337606144

In [4]:
messages = [
    {
        "role": "system",
        "content": "Odpowiadaj krótko, precyzyjnie i wyłącznie w języku polskim.",
    },
    {
        "role": "user",
        "content": """Przeczytaj poniższy artykuł i odpowiedz na pytanie na podstawie treści artykułu. 

Artykuł: Złota Piłka (fr. Ballon d’Or, ang. The Golden Ball) – coroczny plebiscyt piłkarski. Trofeum przyznawane piłkarzowi, który zaprezentował się najlepiej w mijającym roku kalendarzowym (od rozgrywek 2021/22 za zakończony sezon). 
Na pomysł przyznawania Złotej Piłki wpadł szef „France Football” Gabriel Hanot, który w 1956 roku poprosił kolegów z redakcji o głosowanie na najlepszego gracza w Europie. Pierwszym zwycięzcą plebiscytu został Stanley Matthews grający wówczas w Blackpool[1]. 
Początkowo dziennikarze mogli głosować tylko na graczy pochodzących z Europy i dlatego też ani Diego Maradona, ani Pelé (uważani za jednych z najlepszych piłkarzy wszech czasów) nie mieli szans na otrzymanie tej nagrody[2]. 
W 1995 postanowiono zmienić przepisy i umożliwić udział w plebiscycie również graczom spoza Starego Kontynentu, którzy grają w europejskich klubach. Po zmianie przepisów przyznano po jednej honorowej Złotej Piłce dwóm piłkarzom: Diego Maradonie w 1995 oraz Pele w 2013 roku[3]. 
W 2016 dokonano wirtualnego przeliczenia historycznych dokonań piłkarzy i stwierdzono, że w 12 przypadkach Złota Piłka powinna zostać przyznana innemu piłkarzowi, gdyby wszyscy gracze świata mogliby ją otrzymać. 
Uznano, że 7 Złotych Piłek powinien otrzymać Pele (1958, 1959, 1960, 1961, 1963, 1964, 1970)[4], dwie Maradona (1986, 1990) i po jednej Garrincha (1962), Mario Kempes (1978) i Romario (1994)[5]. 

Pytanie: Jacy laureaci Złotej Piłki są wymienieni w artykule?""",
    },
]

input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")

In [5]:
model_inputs = input_ids.to(device)

In [6]:
generated_ids = model.generate(model_inputs, max_new_tokens=1000, do_sample=True)

In [7]:
decoded = tokenizer.batch_decode(generated_ids)

In [8]:
print(decoded[0])

<s><|im_start|> system
Odpowiadaj krótko, precyzyjnie i wyłącznie w języku polskim.<|im_end|> 
<|im_start|> user
Przeczytaj poniższy artykuł i odpowiedz na pytanie na podstawie treści artykułu. 

Artykuł: Złota Piłka (fr. Ballon d’Or, ang. The Golden Ball) – coroczny plebiscyt piłkarski. Trofeum przyznawane piłkarzowi, który zaprezentował się najlepiej w mijającym roku kalendarzowym (od rozgrywek 2021/22 za zakończony sezon). 
Na pomysł przyznawania Złotej Piłki wpadł szef „France Football” Gabriel Hanot, który w 1956 roku poprosił kolegów z redakcji o głosowanie na najlepszego gracza w Europie. Pierwszym zwycięzcą plebiscytu został Stanley Matthews grający wówczas w Blackpool[1]. 
Początkowo dziennikarze mogli głosować tylko na graczy pochodzących z Europy i dlatego też ani Diego Maradona, ani Pelé (uważani za jednych z najlepszych piłkarzy wszech czasów) nie mieli szans na otrzymanie tej nagrody[2]. 
W 1995 postanowiono zmienić przepisy i umożliwić udział w plebiscycie również graczo